# Colab 运行手册：DeepGlobe 土地覆盖分割

这个 notebook 按顺序执行下面所有 cell 就能跑完整个 project。

**重要：Colab 会断线。** checkpoint / 数据 / 产物全部写到 **Google Drive**，重连后从 Drive 拉回来继续跑，不会丢进度。

**推荐配置：** Colab Pro + A100/V100。免费 T4 也能跑，只是慢（×2–3 倍）。

**时间预算（A100）：** 切片 3 分钟 + 6 个架构 ~8 小时 + 3 个 loss ~5 小时 + 评估 1 小时 ≈ **15 小时 GPU**。免费 Colab 一次最多 12h，需要分段跑。

## 0. 检查 GPU

In [ ]:
!nvidia-smi

期望看到 `Tesla T4` / `V100` / `A100`。如果看到 `command not found`，菜单 `Runtime → Change runtime type → GPU`。

## 1. 挂载 Google Drive

会弹出授权窗口，用你的 Google 账号登录。挂载后 Drive 根目录在 `/content/drive/MyDrive/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 所有持久化内容都放这里。断线重连后还在。
import os
DRIVE_ROOT = '/content/drive/MyDrive/landcover-seg'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/raw',         exist_ok=True)  # 原始 DeepGlobe 数据（~2.5GB）
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)  # 模型 .pt（每个 ~100–300MB，共 ~1.5GB）
os.makedirs(f'{DRIVE_ROOT}/outputs',     exist_ok=True)  # history CSV + report JSON + 图
print('Drive paths ready:', os.listdir(DRIVE_ROOT))

## 2. 获取项目代码（private 仓库）

仓库地址：https://github.com/Jerryzhang258/landcover-seg  （private）

Private 仓库在 Colab 里 clone 需要认证。**二选一**，推荐 A1（最干净，一次授权）。

（嫌麻烦的终极方案：`gh repo edit Jerryzhang258/landcover-seg --visibility public --accept-visibility-change-consequences`，之后 `git clone` 不需要任何 token。）

### 方案 A1：gh CLI 登录（推荐）

下面的 cell 会让你打开一个 URL，贴 8 位 code 完成设备授权。一次就行。

In [ ]:
# Colab 上装 gh
!type -p curl >/dev/null && (type -p gh >/dev/null || (curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | sudo dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg && echo 'deb [arch=amd64 signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main' | sudo tee /etc/apt/sources.list.d/github-cli.list > /dev/null && sudo apt-get update -q && sudo apt-get install -y gh))

# device-flow 登录
!gh auth login --hostname github.com --git-protocol https --web
!gh auth setup-git

In [ ]:
%cd /content
!rm -rf landcover-seg
!gh repo clone Jerryzhang258/landcover-seg
%cd /content/landcover-seg
!ls

### 方案 A2：Personal Access Token

1. https://github.com/settings/tokens/new?scopes=repo → Generate → 复制 `ghp_...`
2. 贴到下面 cell 的 `TOKEN` 变量里，跑完马上把 cell 输出清空，不要把 token 一起 commit

（跑过方案 A1 就跳过这个 cell。）

In [ ]:
# from getpass import getpass
# TOKEN = getpass('paste PAT (input hidden): ')
# %cd /content
# !rm -rf landcover-seg
# !git clone https://{TOKEN}@github.com/Jerryzhang258/landcover-seg.git
# TOKEN = None
# %cd /content/landcover-seg
# !ls

## 3. 装依赖

Colab 已经预装 torch，只需装项目专用包。约 2 分钟。

In [ ]:
!pip install -q segmentation-models-pytorch==0.3.3 albumentations==1.3.1 pyyaml tqdm wandb pytest
!python -c "import torch, segmentation_models_pytorch as smp; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| smp:', smp.__version__)"

## 4. 获取 DeepGlobe 数据

**三选一。** 推荐方案 A（Kaggle API）。

### 方案 A：Kaggle API（推荐，~3 分钟）

1. 去 https://www.kaggle.com/settings/account → "Create New API Token" → 下载 `kaggle.json`
2. 把 `kaggle.json` 上传到 Drive 的 `landcover-seg/` 根目录
3. 跑下面的 cell

In [ ]:
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('/content/drive/MyDrive/landcover-seg/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!pip install -q kaggle
# 下载到 Drive，下次就不用重下
RAW = '/content/drive/MyDrive/landcover-seg/raw'
if not os.path.exists(f'{RAW}/train') or len(os.listdir(f'{RAW}/train')) < 100:
    !kaggle datasets download -d balraj98/deepglobe-land-cover-classification-dataset -p /content/ --unzip
    # Kaggle 上的目录结构：/content/train/{id}_sat.jpg, {id}_mask.png
    !mkdir -p {RAW}/train
    !mv /content/train/* {RAW}/train/
    print('downloaded')
else:
    print('already on Drive, skipping download')

!ls {RAW}/train | wc -l

期望看到 `1606`（803 对 × 2 = 1606 个文件）。

### 方案 B：从 Drive 直接读（如果你已经把数据放 Drive 了）

把 DeepGlobe 的 `train/` 目录拷到 Drive 的 `landcover-seg/raw/train/` 下即可，这一节不用跑。

### 方案 C：官方 CodaLab（需要账号）

手动下载后同方案 B。

## 5. 把 data/raw 和 checkpoints 链接到 Drive

关键一步：让 `data/raw/train`, `checkpoints/`, `outputs/` 三个目录**物理上**指向 Drive，这样断线重连后恢复本 notebook 到第 5 步就能接着跑。

`data/tiles/` 不链接 Drive——切片后的 tile 约 5GB，读 I/O 高，放在 Colab 本地 SSD 更快。断线后从 Drive 重跑 `prepare_tiles.py` 即可（3 分钟）。

In [ ]:
%cd /content/landcover-seg
!mkdir -p data
!rm -rf data/raw checkpoints outputs
!ln -sf /content/drive/MyDrive/landcover-seg/raw         data/raw
!ln -sf /content/drive/MyDrive/landcover-seg/checkpoints checkpoints
!ln -sf /content/drive/MyDrive/landcover-seg/outputs     outputs
!ls -la data/ checkpoints outputs

## 6. 单元测试（30 秒）

In [ ]:
!pytest -q

## 7. 切片 + 划分（约 3 分钟）

生成的 tile 放在 Colab 本地 `/content/landcover-seg/data/tiles/`（不进 Drive，因为 5GB 大+ I/O 高）。

In [ ]:
!python scripts/prepare_tiles.py
!ls data/tiles/train/images | wc -l   # 期望 5058
!ls data/tiles/val/images   | wc -l   # 期望 1089
!ls data/tiles/test/images  | wc -l   # 期望 1080

## 8. （可选）登录 wandb

跳过的话 `configs/base.yaml` 把 `use_wandb: true` 改成 `false`，或下面这个 cell 直接把环境变量关了。

In [ ]:
# 要用 wandb：
# import wandb
# wandb.login()   # 会让你贴 API key

# 不用 wandb：
import os
os.environ['WANDB_MODE'] = 'disabled'

## 9. EDA

In [ ]:
!python scripts/plot_eda.py
from IPython.display import Image
Image('outputs/figs/class_dist.png')

In [ ]:
Image('outputs/figs/samples.png')

## 10. 冒烟测试一个模型（T4 约 2h，A100 约 40 min）

**一定要先跑完这个再批量**，确保链路没问题。

In [ ]:
!python -m src.train --config configs/unet_resnet34.yaml

看到 `[done] best mIoU=0.6x  ...` 就算跑通。checkpoint 已经落 Drive，可以在 `/content/drive/MyDrive/landcover-seg/checkpoints/` 看到 `unet_r34_ce_best.pt`。

## 11. 断线恢复（任何时刻断了重连都跑这个）

Colab 免费版一次最多 12h，经常撑不完全部实验。断线后新开 notebook 从 0 走到第 7 步（挂 Drive + clone + 装依赖 + 链接 Drive + 切片），然后用下面的 cell 只训没完成的 run——checkpoint 已经在 Drive，训过的会被自动跳过。

In [ ]:
!ls checkpoints/

In [ ]:
import os
done = {f.replace('_best.pt','') for f in os.listdir('checkpoints') if f.endswith('_best.pt')}
print('already done:', done)

all_runs = [
    ('vanilla_unet',    'vanilla_unet_ce'),
    ('unet_scratch',    'unet_scratch_ce'),
    ('unet_resnet34',   'unet_r34_ce'),
    ('deeplab_r50',     'deeplab_r50_ce'),
    ('deeplab_r101',    'deeplab_r101_ce'),
    ('attn_unet',       'attn_unet_ce'),
]
todo = [cfg for cfg, run in all_runs if run not in done]
print('to train:', todo)

for cfg in todo:
    !python -m src.train --config configs/{cfg}.yaml

## 12. Loss 对比（Phase 5）

先看 Phase 4 的 winner：

In [ ]:
!python scripts/aggregate_results.py
import pandas as pd
df = pd.read_csv('outputs/results.csv')
df = df.sort_values('best_val_mIoU', ascending=False)
df[['run_name','model','loss','params_M','best_val_mIoU','train_time_hr']]

In [ ]:
# 默认假设 winner 是 deeplab_r50。如果不是，改下面的 configs。
loss_runs = ['deeplab_r50_dice', 'deeplab_r50_focal', 'deeplab_r50_combined']

done = {f.replace('_best.pt','') for f in os.listdir('checkpoints') if f.endswith('_best.pt')}
for cfg in loss_runs:
    if cfg in done:
        print(f'skip {cfg} (done)'); continue
    !python -m src.train --config configs/{cfg}.yaml

## 13. 全分辨率评估（Phase 6，约 1 小时 on A100）

In [ ]:
!bash scripts/eval_all.sh

In [ ]:
# 最终结果表
import pandas as pd
df = pd.read_csv('outputs/results.csv')
df = df.sort_values('test_mIoU', ascending=False) if 'test_mIoU' in df.columns else df
df

## 14. 出图

In [ ]:
# 架构对比
!python scripts/plot_curves.py \
    --runs vanilla_unet_ce unet_scratch_ce unet_r34_ce deeplab_r50_ce deeplab_r101_ce attn_unet_ce \
    --out outputs/figs/curves_archs.png

!python scripts/plot_results.py \
    --runs vanilla_unet_ce unet_scratch_ce unet_r34_ce deeplab_r50_ce deeplab_r101_ce attn_unet_ce \
    --out-prefix outputs/figs/archs

from IPython.display import Image, display
display(Image('outputs/figs/curves_archs.png'))
display(Image('outputs/figs/archs_metrics.png'))
display(Image('outputs/figs/archs_perclass.png'))

In [ ]:
# loss 对比
!python scripts/plot_curves.py \
    --runs deeplab_r50_ce deeplab_r50_dice deeplab_r50_focal deeplab_r50_combined \
    --out outputs/figs/curves_losses.png

!python scripts/plot_results.py \
    --runs deeplab_r50_ce deeplab_r50_dice deeplab_r50_focal deeplab_r50_combined \
    --out-prefix outputs/figs/losses

display(Image('outputs/figs/curves_losses.png'))
display(Image('outputs/figs/losses_metrics.png'))

In [ ]:
# 定性对比：从 test split 挑 3 张，看不同模型的预测
import json
splits = json.load(open('data/tiles/splits.json'))
test_ids = splits['test'][:3]
print('using test ids:', test_ids)

!python scripts/visualize_predictions.py \
    --pred-dirs outputs/unet_r34_ce outputs/deeplab_r50_ce outputs/attn_unet_ce \
    --ids {' '.join(test_ids)} \
    --out outputs/figs/qualitative.png

display(Image('outputs/figs/qualitative.png'))

## 15. 下载结果

所有产物已经在 Drive 的 `landcover-seg/outputs/` 里。也可以打包下载：

In [ ]:
!zip -r /content/results.zip outputs/figs outputs/results.csv outputs/*_summary.json outputs/*_history.csv
from google.colab import files
files.download('/content/results.zip')

---

## 常见 Colab 坑

| 问题 | 解决 |
|---|---|
| 12h 超时断线 | 按第 11 节继续 |
| 空闲断线（90 分钟没活动） | Pro 订阅 / 打开浏览器别让标签页睡 |
| `No GPU available` | Runtime → Change runtime type → GPU |
| `CUDA out of memory` | `configs/deeplab_r101.yaml` 里 `bs: 2`；或加 `--grid 4 --tile-size 612` 重新切片 |
| 磁盘满 | `!df -h /content`；把 `/content/landcover-seg/data/tiles` 清一下，需要时重切 |
| Drive 配额超限 | checkpoint 太多，删掉不要的 `.pt`（备份前先 `!cp` 到别处） |
| pip 装完 import 报错 | Runtime → Restart runtime |
| clone 仓库报 403 | 方案 A1 的 `gh auth login` 失败了，重跑那个 cell；或改用方案 A2 的 PAT |